# Colab Inference

This notebook bootstraps the environment and runs the Arc2Face inference pipeline.

It supports both:
- folder mode: sample a subset of identity folders and images
- single image mode: run one image through the pipeline and save a comparison panel

The notebook writes a run-specific output folder, so each execution stays separated.

## Where to place files

Put the files here in Google Drive before running:

- Dataset: `/content/drive/MyDrive/emb2face/data/webface_112x112/`.
  - Keep the dataset structure the same as the training data: one identity folder per person, images inside each folder.
- Adapter checkpoint: `/content/drive/MyDrive/emb2face/outputs/webface_arcada_adapter/models_full/best_linear_adapter.pt`.
  - If you trained the MLP adapter instead, place `best_mlp_adapter.pt` in the same folder.
- Optional persistent caches:
  - InsightFace cache: `/content/drive/MyDrive/emb2face/.insightface/`
  - Arc2Face cache: `/content/drive/MyDrive/emb2face/.cache/arc2face_models/`

If you want to use different paths, edit the variables in the next cell.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Choose the mode you want to run.
# - 'folder' samples N identity folders and K images per identity.
# - 'single' runs one image and saves the comparison panel.
MODE = 'folder'

# Folder mode settings.
NUM_IDENTITIES = 5
IMAGES_PER_IDENTITY = 1
SAVE_COMPARISON_FIGURES = True

# Single-image mode setting.
INPUT_IMAGE = '/content/drive/MyDrive/emb2face/data/webface_112x112/id_160/160_20139.jpg'

# Shared paths.
DRIVE_ROOT = Path('/content/drive/MyDrive/emb2face')
DATASET_ROOT = DRIVE_ROOT / 'data' / 'webface_112x112'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs' / 'webface_arcada_adapter'
INSIGHT_ROOT = DRIVE_ROOT / '.insightface'
ARC2FACE_CACHE_DIR = DRIVE_ROOT / '.cache' / 'arc2face_models'

# Optional seed. Set to an integer if you want repeatable Arc2Face generation.
# Leave as None for fresh random generation each run.
SEED = None

ADAPTER_RUN_MODE = 'full'
DEVICE = 'cuda'
CONFIG_NAME = 'colab_inference.yaml'
COLAB_CONFIG_PATH = None

def _find_repo_root(start: Path):
    for candidate in [start.resolve()] + list(start.resolve().parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'emb2face').exists():
            return candidate
    return None

repo_root = _find_repo_root(Path.cwd())
if repo_root is None:
    repo_root = Path('/content/emb2face')
    repo_url = os.environ.get('EMB2FACE_REPO_URL', 'https://github.com/charan-v2/emb2face.git')
    if not repo_root.exists():
        subprocess.check_call(['git', 'clone', repo_url, str(repo_root)])

os.chdir(repo_root)
sys.path.insert(0, str(repo_root / 'src'))

from emb2face.bootstrap import prepare_environment
repo_root = prepare_environment(
    repo_url=os.environ.get('EMB2FACE_REPO_URL', 'https://github.com/charan-v2/emb2face.git'),
    target_dir=repo_root,
    install=True,
)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

print('Repo root:', repo_root)
print('Drive root:', DRIVE_ROOT)
print('Dataset root:', DATASET_ROOT)
print('Output root:', OUTPUT_ROOT)
print('InsightFace cache:', INSIGHT_ROOT)
print('Arc2Face cache:', ARC2FACE_CACHE_DIR)


In [ ]:
from pathlib import Path
import yaml

config_dir = repo_root / 'config'
config_dir.mkdir(parents=True, exist_ok=True)
COLAB_CONFIG_PATH = config_dir / CONFIG_NAME

with open(repo_root / 'config' / 'default.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg.update({
    'dataset_root': str(DATASET_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'device': DEVICE,
    'seed': SEED,
    'adapter_run_mode': ADAPTER_RUN_MODE,
    'insight_root': str(INSIGHT_ROOT),
    'arc2face_local_dir': str(ARC2FACE_CACHE_DIR),
    'save_comparison_figures': SAVE_COMPARISON_FIGURES,
})

with open(COLAB_CONFIG_PATH, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Wrote config to:', COLAB_CONFIG_PATH)
print('Adapter checkpoint expected at:')
print(OUTPUT_ROOT / f'models_{ADAPTER_RUN_MODE}' / 'best_linear_adapter.pt')


In [ ]:
adapter_ckpt = OUTPUT_ROOT / f'models_{ADAPTER_RUN_MODE}' / 'best_linear_adapter.pt'

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f'Dataset not found at {DATASET_ROOT}. Place your identity-folder dataset there first.'
    )

if MODE == 'folder' and not adapter_ckpt.exists():
    raise FileNotFoundError(
        f'Adapter checkpoint not found at {adapter_ckpt}. Put best_linear_adapter.pt there, or change ADAPTER_RUN_MODE if needed.'
    )

if MODE == 'single' and not Path(INPUT_IMAGE).exists():
    raise FileNotFoundError(f'Input image not found: {INPUT_IMAGE}')

print('Everything looks in place.')
print('Dataset root:', DATASET_ROOT)
print('Adapter checkpoint:', adapter_ckpt)
if MODE == 'single':
    print('Single image:', INPUT_IMAGE)


In [ ]:
from emb2face.cli import main

args = ['infer', '--config', str(COLAB_CONFIG_PATH), '--output-dir', str(OUTPUT_ROOT / 'inference_full')]

if MODE == 'folder':
    args += [
        '--input-dir', str(DATASET_ROOT),
        '--num-identities', str(NUM_IDENTITIES),
        '--images-per-identity', str(IMAGES_PER_IDENTITY),
    ]
    if SAVE_COMPARISON_FIGURES:
        args.append('--save-comparison-figures')
    else:
        args.append('--no-save-comparison-figures')
elif MODE == 'single':
    args += ['--input-image', str(INPUT_IMAGE)]
else:
    raise ValueError("MODE must be either 'folder' or 'single'")

if SEED is not None:
    args += ['--seed', str(SEED)]

print('Running:', ' '.join(args))
main(args)


## Outputs

Each run writes to its own timestamped subfolder under:

- `/content/drive/MyDrive/emb2face/outputs/webface_arcada_adapter/inference_full/`

Inside the run folder you will find:
- `selected_samples.csv`
- `inference_report.csv`
- `summary.csv`
- `reconstructions/`
- `figures/` if comparison figures were enabled

If you switch `MODE` to `single`, the notebook will generate one comparison panel for the selected image.